In [ ]:
# Setup. On Colab (GPU runtime): get the code from GitHub, install the requirements, mount Google Drive and copy the
# data from "My Drive/miniGPT/" (raw/ and splits/). Checkpoints and history.json are saved to Drive. Locally: imports only.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/minigpt").exists():
        !GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/mtf12/minigpt.git /content/minigpt
    %cd /content/minigpt/models
    !pip install -q -r ../requirements.txt
    sys.path.insert(0, str(Path.cwd() / "training"))
    import colab_runner
    colab_runner.setup()

sys.path.insert(0, str(Path.cwd() / "preprocessing"))
sys.path.insert(0, str(Path.cwd() / "tokenizer"))
sys.path.insert(0, str(Path.cwd() / "gpt"))
sys.path.insert(0, str(Path.cwd() / "training"))
import config
import data_exploration   # 1
import clean_dataset      # 2, 3 (uses light_cleaning and heavy_cleaning)
import quality_report     # 4
import balance_languages  # 5
import split_data         # 6
import train_tokenizer    # 7
import evaluate_tokenizer # 8
import encode_text        # 9
import run_gpt            # 10
import train_model        # 11
import training_report    # 12, 13

In [ ]:
# 1) Exploration report on the raw data (raw/ is never modified)
data_exploration.explore("raw")

In [ ]:
# 2) Light cleaning (normalization + basic): raw/ -> datasets/cleaned/
light_stats = clean_dataset.run_light_cleaning()

In [ ]:
# 3) Heavy cleaning (structural) on datasets/cleaned/, then exact duplicates removed
clean_stats = clean_dataset.run_heavy_cleaning(light_stats)

In [ ]:
# 4) Quality report: raw vs. cleaned
quality_report.run(clean_stats)

In [ ]:
# 5) Give Arabic and English about the same number of tokens; the extra articles are deleted
balance_languages.run()

In [ ]:
# 6) Split 80/10/10 per language, mix ar + en, and save to datasets/splits/
split_data.run()

In [ ]:
# 7) Train the joint Arabic + English tokenizer on datasets/splits/train.parquet -> tokenizer/artifacts/tokenizer.json
tokenizer = train_tokenizer.train()

In [ ]:
# 8) Measure it on the validation split: tokens per character for each language
tokenizer_metrics = evaluate_tokenizer.evaluate_saved_tokenizer()

In [ ]:
# 9) Live example: encode a sentence into tokens, then decode the tokens back into the same sentence
encode_text.show_examples(["القاهرة هي عاصمة مصر وأكبر مدينة في العالم العربي.", "The Nile is the longest river in Africa."])

In [ ]:
# 10) Build the GPT, print its size, run a real validation batch and generate samples
run_gpt.main()

In [ ]:
# 11) Train the GPT on train, check validation after every epoch and save checkpoints.
#     If a checkpoint exists (a dropped Colab session), training continues from it.
history = train_model.run_training(resume=True)

In [ ]:
# 12) Loss curves: train loss of every update and validation loss after every epoch, from history.json
training_report.plot_history()

In [ ]:
# 13) Generate text with the best model (the epoch with the lowest validation loss)
training_report.generate_samples(["القاهرة هي", "تعتبر اللغة العربية", "The Nile is", "In the year 1900,"])